In [8]:
import glob
import io
import os
import shutil
import sys
import time
import pathlib
import re
from collections import OrderedDict
from datetime import datetime, timedelta
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import polars as pl
import pyautogui
import win32clipboard

from PIL import Image, ImageGrab

from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    NoAlertPresentException,
    NoSuchElementException,
    TimeoutException,
    StaleElementReferenceException
)

In [9]:
def input_data(data_dir):
    list_files = []
    for path in pathlib.Path(data_dir).glob('**/*.*'):
        if path.suffix.lower() in ['.xlsx', '.csv'] and path.stat().st_size > 0:
            export_time = datetime.fromtimestamp(path.stat().st_mtime)
            try:
                df = pl.read_excel(path) if path.suffix.lower() == '.xlsx' else pl.read_csv(path, infer_schema_length=10000, ignore_errors=True)
                if not df.is_empty():
                    df = df.with_columns([
                        pl.lit(path.stem).alias('sheet_name'),
                        pl.lit(export_time).alias('Export time')
                    ])
                    list_files.append(df)
            except:
                continue
                
    return pl.concat(list_files, how="diagonal_relaxed") if list_files else pl.DataFrame()

In [10]:
first_glob = os.path.expanduser("~").replace("\\", "/")

folder_paths = {
    "req": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OU.xlsx",
    "current_agents": f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/CAPTURE/current_agent/",
    "input_iex_intervals":f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS',

}

In [11]:
def process_ou_hours(file_path):
    try:
        df_raw = pl.read_excel(source=file_path, has_header=False, infer_schema_length=0)
        
        header_vals = df_raw.row(0)
        new_columns = [
            val.strftime("%Y-%m-%d") if isinstance(val, datetime) else str(val).strip() if val is not None else "Unknown" 
            for val in header_vals
        ]
        df_raw.columns = new_columns
        
        df = df_raw.slice(1).with_columns(
            pl.col("Interval").cast(pl.String).str.replace(r"^.*1899-12-31\s+", "").str.slice(0, 8).alias("Interval")
        )
        
        final_df = (df.unpivot(index=["LOB", "Site", "Interval"], variable_name="Date_Str", value_name="Value")
            .with_columns([
                (pl.col("Date_Str").str.slice(0, 10) + " " + pl.col("Interval"))
                .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
                .dt.truncate("1m").alias("PST_Datetime"),
                
                pl.col("Value").cast(pl.Float64, strict=False).fill_null(0.0).alias("OU Req by Site"),
                pl.col("LOB").str.strip_chars(), 
                pl.col("Site").str.strip_chars()
            ])
            .with_columns(
                pl.col("PST_Datetime")
                .dt.replace_time_zone("America/Los_Angeles", non_existent="null", ambiguous="earliest")
                .alias("_tz_aware")
            )
            .with_columns([
                pl.col("_tz_aware").dt.convert_time_zone("Asia/Ho_Chi_Minh").dt.replace_time_zone(None).alias("VNT_Datetime"),
                pl.col("_tz_aware").dt.convert_time_zone("Africa/Cairo").dt.replace_time_zone(None).alias("CLT_Datetime"),
                pl.col("_tz_aware").dt.convert_time_zone("Asia/Kolkata").dt.replace_time_zone(None).alias("IST_Datetime"),
                pl.col("OU Req by Site").sum().over(["LOB", "PST_Datetime"]).alias("Total OU Req")
            ])
            .drop("_tz_aware")
        )
        return final_df.select(["LOB", "Site", "PST_Datetime", "VNT_Datetime", "CLT_Datetime", "IST_Datetime", "Total OU Req", "OU Req by Site"])
    except Exception as e:
        print(f"Error processing file: {e}")
        return pl.DataFrame()

req_path = os.path.join(first_glob, r"Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\OU.xlsx")
df_req = process_ou_hours(folder_paths["req"])
df_req

LOB,Site,PST_Datetime,VNT_Datetime,CLT_Datetime,IST_Datetime,Total OU Req,OU Req by Site
str,str,datetime[μs],datetime[μs],datetime[μs],datetime[μs],f64,f64
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 00:00:00,2025-12-29 15:00:00,2025-12-29 10:00:00,2025-12-29 13:30:00,4.215,1.76
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 00:30:00,2025-12-29 15:30:00,2025-12-29 10:30:00,2025-12-29 14:00:00,4.215,1.79
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 01:00:00,2025-12-29 16:00:00,2025-12-29 11:00:00,2025-12-29 14:30:00,4.215,2.165
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 01:30:00,2025-12-29 16:30:00,2025-12-29 11:30:00,2025-12-29 15:00:00,4.215,1.96
"""Lodging chat""","""Concentrix (Pune)""",2025-12-29 02:00:00,2025-12-29 17:00:00,2025-12-29 12:00:00,2025-12-29 15:30:00,4.215,2.29
…,…,…,…,…,…,…,…
"""Non-Lodging chat""","""Concentrix (Kolkata)""",2026-05-10 21:30:00,2026-05-11 11:30:00,2026-05-11 07:30:00,2026-05-11 10:00:00,7.08,2.42
"""Non-Lodging chat""","""Concentrix (Kolkata)""",2026-05-10 22:00:00,2026-05-11 12:00:00,2026-05-11 08:00:00,2026-05-11 10:30:00,9.065,2.855
"""Non-Lodging chat""","""Concentrix (Kolkata)""",2026-05-10 22:30:00,2026-05-11 12:30:00,2026-05-11 08:30:00,2026-05-11 11:00:00,9.735,3.075


In [12]:
IEX = input_data(folder_paths["input_iex_intervals"])
IEX

,Month,Week_Monday,Date_Converted,Employee Name,OracleID,IEX ID,Email Id,First Shift,Scheduled Activity,VNT_Intervals,PST_Intervals,VNT_Interval_Range,PST_Interval_Range,Datetime_Start_Time,Datetime_End_Time,Duration,Work Category,sheet_name,Export time
i64,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,f64,str,str,datetime[μs]
0,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Open Time""","""2025-12-01 21:00:00""","""2025-12-01 06:00:00""","""21:00-21:30""","""06:00-06:30""","""2025-12-01 21:00:00""","""2025-12-01 21:30:00""",0.5,"""Productive""","""2025-12-01""",2026-04-07 10:30:58
1,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Open Time""","""2025-12-01 21:30:00""","""2025-12-01 06:30:00""","""21:30-22:00""","""06:30-07:00""","""2025-12-01 21:30:00""","""2025-12-01 22:00:00""",0.5,"""Productive""","""2025-12-01""",2026-04-07 10:30:58
2,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Open Time""","""2025-12-01 22:00:00""","""2025-12-01 07:00:00""","""22:00-22:30""","""07:00-07:30""","""2025-12-01 22:00:00""","""2025-12-01 22:15:00""",0.25,"""Productive""","""2025-12-01""",2026-04-07 10:30:58
3,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Break_1""","""2025-12-01 22:00:00""","""2025-12-01 07:00:00""","""22:00-22:30""","""07:00-07:30""","""2025-12-01 22:15:00""","""2025-12-01 22:30:00""",0.25,"""Unproductive""","""2025-12-01""",2026-04-07 10:30:58
4,"""Dec-25""","""2025-12-01""","""2025-12-01""","""HOANG THI PHUONG HOA""",102484482,3000182,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Open Time""","""2025-12-01 22:30:00""","""2025-12-01 07:30:00""","""22:30-23:00""","""07:30-08:00""","""2025-12-01 22:30:00""","""2025-12-01 23:00:00""",0.5,"""Productive""","""2025-12-01""",2026-04-07 10:30:58
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
37623,"""May-26""","""2026-05-04""","""2026-05-10""","""DUONG THI THUY DUONG""",103305150,3112809,"""thithuyduong.duong@concentrix.…","""2000-0500""","""Open Time""","""2026-05-11 03:00:00""","""2026-05-10 13:00:00""","""03:00-03:30""","""13:00-13:30""","""2026-05-11 03:00:00""","""2026-05-11 03:15:00""",0.25,"""Productive""","""2026-05-04""",2026-05-07 10:22:14
37624,"""May-26""","""2026-05-04""","""2026-05-10""","""DUONG THI THUY DUONG""",103305150,3112809,"""thithuyduong.duong@concentrix.…","""2000-0500""","""Break_2""","""2026-05-11 03:00:00""","""2026-05-10 13:00:00""","""03:00-03:30""","""13:00-13:30""","""2026-05-11 03:15:00""","""2026-05-11 03:30:00""",0.25,"""Unproductive""","""2026-05-04""",2026-05-07 10:22:14
37625,"""May-26""","""2026-05-04""","""2026-05-10""","""DUONG THI THUY DUONG""",103305150,3112809,"""thithuyduong.duong@concentrix.…","""2000-0500""","""Open Time""","""2026-05-11 03:30:00""","""2026-05-10 13:30:00""","""03:30-04:00""","""13:30-14:00""","""2026-05-11 03:30:00""","""2026-05-11 04:00:00""",0.5,"""Productive""","""2026-05-04""",2026-05-07 10:22:14


In [13]:
REQ_SITE_MAPPING = {
    "Concentrix (Pune)": "PUN",
    "Concentrix (Ho Chi Minh City)": "HCM",
    "Concentrix (Kolkata)": "KOL",
    "Concentrix (Cairo)": "CAI"
}

SITE_MAPPING = {"Pune": "PUN", "Ho Chi Minh": "HCM", "Kolkata": "KOL", "Cairo": "CAI"}

LOB_MAPPING = {
    "Lodging Chat": ["Chat_OD_EN_Car_Activity", "Chat_OD_EN_Lodging", "Chat - Global English Lodging Nesting", "Chat_Lodging English w Car"],
    "Non Lodging Chat": ["Chat - Global English Non- Lodging Nesting", "Chat_OD_EN_Dual_GDS"]
}

ACTIVE_STATES = ["AVAILABLECHAT", "OUTBOUNDCALL"]

outage_db = (
    input_data(folder_paths["current_agents"])
    .filter(
        pl.col("Export time").is_in(
            input_data(folder_paths["current_agents"])
            .select("Export time").unique().sort("Export time", descending=True).limit(16).get_column("Export time")
        )
    )
    .select([
        "Business Location", "Export time", "Agent Name", "Agent Manager",
        "Connect State", "Assigned Workitem Count", "Duration", "Queue Group / Routing Profile"
    ])
    .with_columns([
        pl.col("Export time").cast(pl.Datetime, strict=False).dt.truncate("30m").alias("VN_Datetime"),
        pl.col("Duration").str.strptime(pl.Time, "%H:%M:%S", strict=False)
    ])
    .with_columns([
        pl.col("VN_Datetime").dt.replace_time_zone("Asia/Ho_Chi_Minh").dt.convert_time_zone("America/Los_Angeles").dt.replace_time_zone(None).alias("PST_Datetime")
    ])
    .with_columns([
        pl.when(pl.col("Queue Group / Routing Profile").is_in(LOB_MAPPING["Lodging Chat"])).then(pl.lit("Lodging Chat"))
        .when(pl.col("Queue Group / Routing Profile").is_in(LOB_MAPPING["Non Lodging Chat"])).then(pl.lit("Non Lodging Chat"))
        .otherwise(None).alias("LOB")
    ])
    .filter(pl.col("LOB").is_not_null())
)

active_cond = pl.col("Connect State").is_in(ACTIVE_STATES) | (pl.col("Assigned Workitem Count").cast(pl.Int32, strict=False).fill_null(0) > 0)

inactive_cond = ~active_cond

def add_time_cols(df, col="PST_Datetime"):
    return (
        df.with_columns(pl.col(col).dt.replace_time_zone("America/Los_Angeles", non_existent="null", ambiguous="earliest").alias("_tz"))
        .with_columns([
            pl.col(col).dt.date().alias("PST_Date"),
            pl.col(col).dt.strftime("%H:%M").alias("PST"),
            pl.col("_tz").dt.convert_time_zone("Asia/Kolkata").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("IST"),
            pl.col("_tz").dt.convert_time_zone("Africa/Cairo").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("CLT"),
            pl.col("_tz").dt.convert_time_zone("Asia/Ho_Chi_Minh").dt.replace_time_zone(None).dt.strftime("%H:%M").alias("VNT")
        ])
        .drop("_tz")
    )

def build_hc(df, cond, label):

    value_col = f"Heads {label}"

    return (
        df.group_by(["LOB", "PST_Datetime"])
        .agg([
            pl.col("Agent Name").filter(pl.col("Business Location").str.contains(loc) & cond).n_unique().alias(prefix)
            for loc, prefix in SITE_MAPPING.items()
        ])
        .fill_null(0)
        .sort(["LOB", "PST_Datetime"])
        .unpivot(index=["LOB", "PST_Datetime"], on=list(SITE_MAPPING.values()), variable_name="Site", value_name=value_col)
        .pipe(add_time_cols)
        .select(["LOB", "Site", value_col, "PST_Datetime", "PST_Date", "PST", "IST", "CLT", "VNT"])
        .sort(["LOB", "Site"])
    )

df_active = build_hc(outage_db, active_cond, "Active")

df_inactive = build_hc(outage_db, inactive_cond, "Inactive")

df_req = (
    df_req
    .with_columns([
        pl.col("Site").replace(REQ_SITE_MAPPING),

        pl.when(pl.col("LOB") == "Lodging chat").then(pl.lit("Lodging Chat"))
        .when(pl.col("LOB") == "Non-Lodging chat").then(pl.lit("Non Lodging Chat"))
        .otherwise(pl.col("LOB"))
        .alias("LOB")
    ])
    .select(["LOB", "Site", "PST_Datetime", "Total OU Req", "OU Req by Site"])
)

df_active = (
    df_active
    .join(df_req, on=["LOB", "Site", "PST_Datetime"], how="left")
    .with_columns([
    (pl.col("Heads Active") - pl.col("OU Req by Site")).round(1).alias("Missing Heads"),

    pl.when(
        (pl.col("OU Req by Site").is_null()) |
        (pl.col("OU Req by Site") == 0)
    )
    .then(pl.lit("-"))
    .otherwise(
        (((pl.col("Heads Active") / pl.col("OU Req by Site")) * 100).round(1).cast(pl.String) + "%")
    )
    .alias("Heads Active / OU Req by Site"),

    pl.col("Total OU Req").round(1),
    pl.col("OU Req by Site").round(1)
])
    .select([
        "LOB", "Site", "Total OU Req", "OU Req by Site", "Heads Active",
        "Missing Heads", "Heads Active / OU Req by Site",
        "PST_Datetime", "PST_Date", "PST", "IST", "CLT", "VNT"
    ])
)

df_inactive_details = (
    outage_db
    .filter(inactive_cond)
    .select(["Agent Name", "LOB", "Business Location", "Connect State", "Duration", "PST_Datetime"])
    .rename({"PST_Datetime": "PST_Interval"})
    .pipe(add_time_cols, "PST_Interval")
    .drop("PST_Interval")
)

latest_pst = df_inactive_details.select(pl.col("PST").max()).item()

df_active = df_active.filter(pl.col("PST") == latest_pst)

df_inactive = df_inactive.filter(pl.col("PST") == latest_pst)

df_lodging = df_inactive_details.filter(pl.col("LOB") == "Lodging Chat").sort(["Business Location", "Agent Name"])

df_non_lodging = df_inactive_details.filter(pl.col("LOB") == "Non Lodging Chat").sort(["Business Location", "Agent Name"])

# with pl.Config(tbl_rows=-1):
#     display(df_lodging)
#     display(df_non_lodging)
#     display(df_active)
#     display(df_inactive)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10876\4044761893.py:19: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(


In [14]:
from IPython.display import HTML

SITE_FULLNAME = {"PUN": "Pune", "HCM": "Ho Chi Minh", "KOL": "Kolkata", "CAI": "Cairo"}

df_merge = (
    df_active.join(
        df_inactive.select(["LOB", "Site", "Heads Inactive"]),
        on=["LOB", "Site"],
        how="left"
    )
    .with_columns(pl.col("Site").replace(SITE_FULLNAME))
    .sort(["LOB", "Site"])
)

pdf = df_merge.to_pandas()
lodging_detail = df_lodging.to_pandas()
non_lodging_detail = df_non_lodging.to_pandas()

def build_summary_table(lob_name):

    sub = pdf[pdf["LOB"] == lob_name]

    html = f"""
    <div style="width:49%">
    <h3 style="text-align:center">{lob_name}</h3>

    <table border="1" style="width:100%;border-collapse:collapse;text-align:center;font-family:Arial;font-size:12px">
    <tr style="background:#222;color:white">
    <th>PST Date</th><th>PST</th><th>IST</th><th>CLT</th><th>VNT</th>
    <th>Site</th><th>Total OU Req</th><th>OU Req by Site</th>
    <th>Heads Active</th><th>Heads Inactive</th>
    <th>Missing Heads</th><th>Heads Active / OU Req by Site</th>
    </tr>
    """

    for i, (_, r) in enumerate(sub.iterrows()):

        html += "<tr>"

        if i == 0:

            rowspan = len(sub)

            html += f"""
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST_Date'].strftime('%Y-%m-%d')}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['IST']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['CLT']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['VNT']}</td>
            """

        html += f"""
        <td><b>{r['Site']}</b></td>
        <td>{r['Total OU Req']}</td>
        <td>{r['OU Req by Site']}</td>
        <td>{r['Heads Active']}</td>
        <td>{r['Heads Inactive']}</td>
        <td style="
        background:{
            '#d32f2f' if r['Missing Heads'] < 0
            else '#2e7d32' if r['Missing Heads'] > 0
            else 'transparent'
        };
        color:white;
        font-weight:bold
        ">
        {r['Missing Heads']}
        </td>
        <td>{r['Heads Active / OU Req by Site']}</td>
        </tr>
        """

    html += "</table></div>"

    return html

def build_detail_table(df, title):

    html = f"""
    <div style="width:49%">
    <h3 style="text-align:center">{title} Details</h3>

    <table border="1" style="width:100%;border-collapse:collapse;text-align:center;font-family:Arial;font-size:12px">
    <tr style="background:#222;color:white">
    <th>PST Date</th><th>PST</th><th>IST</th><th>CLT</th><th>VNT</th>
    <th>Agent Name</th><th>Business Location</th><th>Connect State</th><th>Duration</th>
    </tr>
    """

    rowspan = len(df)

    for i, (_, r) in enumerate(df.iterrows()):

        html += "<tr>"

        if i == 0:

            html += f"""
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST_Date'].strftime('%Y-%m-%d')}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['PST']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['IST']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['CLT']}</td>
            <td rowspan="{rowspan}" style="vertical-align:middle">{r['VNT']}</td>
            """

        html += f"""
        <td>{r['Agent Name']}</td>
        <td>{r['Business Location']}</td>
        <td>{r['Connect State']}</td>
        <td>{r['Duration']}</td>
        </tr>
        """

    html += "</table></div>"

    return html

html = f"""
<style>
table {{
    width:100%;
    border-collapse:collapse;
    text-align:center;
    font-family:Arial;
    font-size:13px;
    table-layout:auto;
}}

td, th {{
    padding:6px 10px;
    white-space:nowrap;
}}

th {{
    background:#222;
    color:white;
}}
</style>

<div style="margin-bottom:25px">
{build_summary_table("Lodging Chat")}
</div>

<div style="margin-bottom:40px">
{build_summary_table("Non Lodging Chat")}
</div>

<div style="margin-bottom:25px">
{build_detail_table(lodging_detail, "Lodging Chat")}
</div>

<div>
{build_detail_table(non_lodging_detail, "Non Lodging Chat")}
</div>
"""

display(HTML(html))